Extracting the Taxonomy for MedMCQA dataset

In [1]:
import pandas as pd
import numpy as np
train = pd.read_json('/content/drive/MyDrive/Research Project Synthetic Data/data/train.json', lines=True)
val = pd.read_json('/content/drive/MyDrive/Research Project Synthetic Data/data/dev.json', lines=True)


In [2]:
train = train[['subject_name', 'topic_name']]
val = val[['subject_name', 'topic_name']]

In [3]:
print(len(val))

4183


In [4]:
print(len(train))

182822


SOME ROWS HAVE MISSING TOPCS

In [5]:
print("train topic nulls:", train['topic_name'].isna().sum())
print("valid topic nulls:", val['topic_name'].isna().sum())

train topic nulls: 95613
valid topic nulls: 3760


NORMALISING TOPICS

In [6]:
for df in (train, val):
    df['subject_name'] = df['subject_name'].str.strip().str.lower()
    df['topic_name']   = df['topic_name'].str.strip().str.lower()

Checking if all the subjects in the train and val splits are the same

In [7]:
train['subject_name'].unique()

array(['anatomy', 'biochemistry', 'surgery', 'ophthalmology',
       'physiology', 'social & preventive medicine',
       'gynaecology & obstetrics', 'anaesthesia', 'psychiatry',
       'microbiology', 'medicine', 'pharmacology', 'dental', 'ent',
       'forensic medicine', 'pediatrics', 'orthopaedics', 'radiology',
       'pathology', 'skin', 'unknown'], dtype=object)

In [8]:
val['subject_name'].unique()

array(['physiology', 'medicine', 'biochemistry', 'ophthalmology',
       'anatomy', 'pediatrics', 'pathology', 'dental', 'microbiology',
       'anaesthesia', 'radiology', 'gynaecology & obstetrics',
       'pharmacology', 'social & preventive medicine', 'ent', 'surgery',
       'forensic medicine', 'psychiatry', 'skin', 'orthopaedics',
       'unknown'], dtype=object)


Inspecting missing topic names and inspecting per topic popuation



In [9]:
train['subject_name'] = train['subject_name'].str.strip().str.lower()
train['topic_name']   = train['topic_name'].str.strip().str.lower()

In [10]:
print('rows with null topic:', train['topic_name'].isna().sum())
print('As fraction: ', train['topic_name'].isna().mean())

rows with null topic: 95613
As fraction:  0.5229841047576331


In [11]:
topic_counts = train.groupby(['subject_name','topic_name']).size()
print(topic_counts.describe())
print((topic_counts < 10).mean())

count    2742.000000
mean       31.804887
std        80.942471
min         1.000000
25%         4.000000
50%        11.000000
75%        24.750000
max      1691.000000
dtype: float64
0.47045951859956237


Generating the Taxonomy

In [ ]:
import json
train['subject_name'] = train['subject_name'].str.strip().str.lower()
train['topic_name']   = train['topic_name'].str.strip().str.lower()
val['subject_name'] = val['subject_name'].str.strip().str.lower()

valid_subj_counts = val.groupby('subject_name').size()

taxonomy = {}
for subject, group in train.groupby('subject_name'):
    # real, non-null topics under this subject → generation guidance only
    topics = sorted(group['topic_name'].dropna().unique().tolist())
    taxonomy[subject] = {
        "train_count": int(len(group)),
        "valid_count": int(valid_subj_counts.get(subject, 0)),
        "generation_topics": topics,   # metadata for richer generation, NOT coverage nodes
    }

with open("taxonomy.json", "w") as f:
    json.dump(taxonomy, f, indent=2)

print(f"Subjects (coverage units): {len(taxonomy)}")

Subjects (coverage units): 21


MedMCQA's topic_name column is contaminated with provenance labels (exam names, years) mixed in among the real medical topics which need to be removed from the taxonomy.

In [ ]:
import re

# known exam-name tokens — extend this list as you spot more
EXAM_TOKENS = {"neet", "aiims", "aipmt", "pgi", "jipmer", "pg", "mds", "dnb", "fmge", "exam", "misc.", "miscellaneous"}
KEEP_ALWAYS = {"investigation in ophthalmology and miscellaneous topics", "national immunization schedule 2020-21"}  # the real ones you spotted

def looks_like_provenance(topic):
    t = topic.lower().strip()
    if t in KEEP_ALWAYS:
        return False


    t = topic.lower().strip()
    # contains a 4-digit year
    if re.search(r'\b(19|20)\d{2}\b', t):
        return True
    # is (or starts with) a known exam name
    tokens = set(re.split(r'[\s\-_]+', t))
    if tokens & EXAM_TOKENS:
        return True
    return False

In [ ]:
for subject, data in taxonomy.items():
    kept = [t for t in data['generation_topics'] if not looks_like_provenance(t)]
    removed = [t for t in data['generation_topics'] if looks_like_provenance(t)]
    print(f"\n{subject}: kept {len(kept)}, removed {len(removed)}")
    print("  removed sample:", removed[:10])
    print("  kept sample:", kept[:10])


anaesthesia: kept 45, removed 16
  removed sample: ['all india exam', 'dnb 2018', 'fmge 2017', 'fmge 2018', 'fmge 2019', 'jipmer 2017', 'jipmer 2018', 'jipmer 2019', 'miscellaneous', 'miscellaneous (anesthetic equipment)']
  kept sample: ['airway', 'anaesthesia for special situations', 'anaesthesia of special situations', 'anaesthesia q bank', 'anaesthetic equipments', 'anesthesia circuit', 'anesthesia for cardiovascular disease and surgery', 'anesthesia for kidney disease', 'anesthesia for liver disease', 'anesthesia for neurologic & psychiatric diseases']

anatomy: kept 199, removed 15
  removed sample: ['abdomen: miscellaneous', 'all india exam', 'dnb 2018', 'fmge 2017', 'fmge 2018', 'fmge 2019', 'jipmer 2017', 'jipmer 2018', 'jipmer 2019', 'misc.']
  kept sample: ['abdomen & pelvis', 'abdomen and pelvis', 'abdomen, pelvis and perineum', 'abdominal wall', 'abdominal wall ,inguinal and femoral region', 'abnormal labor', 'aerial supply', 'arm and cubital fossa', 'avascular necrosis a

In [ ]:
# applying the cleaning to each subject's topic list
for subject, data in taxonomy.items():
    data['generation_topics'] = [
        t for t in data['generation_topics']
        if not looks_like_provenance(t)
    ]

# saving
with open("taxonomy.json", "w", encoding="utf-8") as f:
    json.dump(taxonomy, f, indent=2, ensure_ascii=False)

print("Saved cleaned taxonomy.json")

Saved cleaned taxonomy.json


Topics per subject

In [ ]:
import json
BASE = '/content/drive/MyDrive/Research Project Synthetic Data'
with open(f'{BASE}/Taxonomy/taxonomy_filtered.json') as f:
    taxonomy = json.load(f)

for s, d in taxonomy.items():
    print(f"{s:30} {len(d['generation_topics']):4} topics")

total = sum(len(d['generation_topics']) for d in taxonomy.values())
print(f"\nTotal topic-leaves: {total}")

anatomy                         199 topics
biochemistry                     93 topics
dental                           35 topics
ent                              90 topics
forensic medicine                70 topics
gynaecology & obstetrics        225 topics
medicine                        210 topics
microbiology                     86 topics
ophthalmology                   106 topics
pathology                       204 topics
pediatrics                      183 topics
pharmacology                    120 topics
physiology                      111 topics
radiology                        77 topics
social & preventive medicine    157 topics
surgery                         216 topics

Total topic-leaves: 2182
